# 01 - Bronze Ingestion

**Purpose of the Bronze Layer:**  This notebook is the first stage of the TfL BikePoint data pipeline. Pull the current snapshot from the TfL BikePoint API and append the raw response into the Bronze Delta table.

**Source:** `https://api.tfl.gov.uk/BikePoint/` 

**Target table:** `workspace.bikepoint.bronze_bikepoint_raw`

**Underatanding:**
- One row per BikePoint station per API run.
- Append only, each run adds new rows, never overwrites.
- Raw JSON for each station is preserved as a string column, untouched

**Why Bronze Exists:** Bronze is not designed for analysis. It is designed to safely capture the raw API data before any cleaning, parsing, or business logic is applied.

This is important because:
- We keep the original API response
- We can reprocess the data later if there is any transformation logic changes
- We can audit exactly what was received from the source
- Downstream Silver and Gold tables can be rebuilt from Bronze
- Allow us to build history over time

**Why Bronze is Append Only:** This layer is the single source of truth. If there is a mistake in the Silver or Gold logic later, we can rebuild those layers from Bronze without calling the API again. This is why we do not overwrite or delete Bronze data.

In [0]:
%sql
-- Create the project schema if it does not already exist.
CREATE SCHEMA IF NOT EXISTS workspace.bikepoint;

## Configuration

This section defines the main project variables.

In [0]:
# TfL BikePoint API endpoint
TFL_URL = "https://api.tfl.gov.uk/BikePoint/"

# Timeout in seconds for the API request
# This prevents the notebook from hanging forever if the API does not respond.
REQ_TIMEOUT_SEC = 30

CATALOG = "workspace"
SCHEMA = "bikepoint"
BRONZE_TABLE = "bronze_bikepoint_raw"

BRONZE_TABLE_FQN = f"{CATALOG}.{SCHEMA}.{BRONZE_TABLE}"

## Fetch from the TfL BikePoint API

This section calls the TfL BikePoint API using Python. The API returns a live snapshot of BikePoint stations across London. Each station includes information such as station ID, name, location, number of bikes, number of empty docks, and e-bike availability.

Make a single HTTP GET request and capture:
- The raw JSON response 
- Ingestion metadata: timestamp, source URL, HTTP status, station count

In [0]:
import requests
from datetime import datetime, timezone

# Capture the ingestion timestamp in UTC.
ingestion_ts = datetime.now(timezone.utc)

# All rows from this API call will share the same pipeline_run_id.
pipeline_run_id = ingestion_ts.strftime("%Y%m%d_%H%M%S")

# Call the TfL BikePoint API
response = requests.get(TFL_URL, timeout=REQ_TIMEOUT_SEC)
response.raise_for_status()   # Raise an error if the API call fails.

stations = response.json()    # Convert the API response into Python objects. Python list of dicts, one per station
print(f"HTTP {response.status_code} | {len(stations)} stations | fetched at {ingestion_ts.isoformat()}")

## Create the Bronze DataFrame

This section converts the API response into a Spark DataFrame. The Bronze table stores one row per station per pipeline run.

Convert the Python list of station dicts into a Spark DataFrame. Each row will contain:
- `bikepoint_id` - the station ID from TfL.
- `raw_payload` - the full raw JSON for that station as JSON string.
- `ingestion_ts` - when this data was collected.
- `pipeline_run_id` - unique ID for this pipeline run.
- `source_url` - the API endpoint used.
- `http_status` - the HTTP response code from the API.
- `station_count` - total number of stations returned by the API.

Keeping `raw_payload` as a JSON string at this stage means we never lose information. Silver will parse it into typed columns.

In [0]:
import json
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, IntegerType
)

# Create one Bronze row per BikePoint station.
rows = [
    Row(
        bikepoint_id   = station["id"],
        raw_payload    = json.dumps(station),  
        ingestion_ts   = ingestion_ts,
        pipeline_run_id=pipeline_run_id,
        source_url     = TFL_URL,
        http_status    = response.status_code,
        station_count  = len(stations),
    )
    for station in stations
]

# Define the Bronze schema explicitly.
# This is better than letting Spark guess the schema.
bronze_schema = StructType([
    StructField("bikepoint_id",  StringType(),    nullable=False),
    StructField("raw_payload",   StringType(),    nullable=False),
    StructField("ingestion_ts",  TimestampType(), nullable=False),
    StructField("pipeline_run_id", StringType(), nullable=False),
    StructField("source_url",    StringType(),    nullable=False),
    StructField("http_status",   IntegerType(),   nullable=False),
    StructField("station_count", IntegerType(),   nullable=False),
])

# Convert the Python rows into a Spark DataFrame.
bronze_df = spark.createDataFrame(rows, schema=bronze_schema)
print(f"Built DataFrame with {bronze_df.count()} rows")

In [0]:
# Display the schema so we can confirm the column names and data types
bronze_df.printSchema()

## Preview Data

Before writing the data to the table, we preview a few rows.

This helps confirm that:

- The API data was received correctly.
- Each row has a BikePoint ID.
- The raw JSON payload is present.
- The ingestion metadata has been added.

In [0]:
display(bronze_df.limit(5))

## Write to the Bronze Delta table

Write the DataFrame to a Delta table using **append mode** instead of **overwrite**, each notebook run adds rows on top of what's already there. Existing rows are never modified.

We do not use overwrite because we want to keep historical snapshots over time.

In [0]:
# Write the Bronze DataFrame to a Delta table.
# Append mode keeps previous snapshots and adds the new API result.
(
    bronze_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE_TABLE_FQN)
)

## Validate the Bronze Load

After writing the data, we run a simple validation query.

This checks:

- Total number of rows in the Bronze table.
- Number of unique BikePoint stations.
- Number of distinct snapshots collected.
- Latest pipeline run ID.
- Latest ingestion timestamp.

This gives us a quick confidence check that the data was loaded successfully.

In [0]:
# Run a simple validation query against the Bronze table.
result = spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT bikepoint_id) AS unique_stations,
        COUNT(DISTINCT ingestion_ts) AS distinct_snapshots,
        MAX(pipeline_run_id) AS latest_pipeline_run,
        MAX(ingestion_ts) AS latest_snapshot
    FROM {BRONZE_TABLE_FQN}
""")
result.show(truncate=False)